# TRM Compiler Pass Ordering — Real LLVM

Train a 60K-param TRM to order LLVM optimization passes, then benchmark on real CompilerGym.

**Runtime:** Change runtime type → T4 GPU.

In [ ]:
!pip install -q compiler_gym torch numpy matplotlib

In [ ]:
import os

# CHANGE THIS to your repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/trm-youtubevids.git'
PROJECT_DIR = '/content/trm-youtubevids'

if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
else:
    !cd $PROJECT_DIR && git pull

os.chdir(PROJECT_DIR)

## Config

In [ ]:
import torch
from pathlib import Path

CONFIG = {
    'benchmarks': [
        'qsort', 'adpcm', 'blowfish', 'bzip2', 'dijkstra', 'sha',
        'gsm', 'ispell', 'jpeg-c', 'lame', 'patricia', 'rijndael',
        'stringsearch', 'susan', 'tiff2bw', 'tiff2rgba', 'tiffdither', 'tiffmedian'
    ],
    'episodes_per_benchmark': 50,
    'max_steps_per_episode': 30,
    'latent_dim': 64,
    'hidden_dim': 128,
    'n_recursions': 6,
    'epochs': 30,
    'batch_size': 64,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'eval_benchmarks': ['qsort', 'adpcm', 'blowfish', 'bzip2', 'dijkstra', 'sha'],
    'eval_max_steps': 30,
    'seed': 42,
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('trm_compiler_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Generate Traces (Real LLVM)

Runs real LLVM passes via CompilerGym on 18 cbench benchmarks.
~30-60 min first time. Cached after that.

In [ ]:
import time
from trm_compiler.data import generate_compiler_traces, load_traces, CompilerTraceDataset
from trm_compiler.env_wrapper import NUM_PASSES

TRACES_PATH = str(OUTPUT_DIR / 'traces_real_llvm.json')

if Path(TRACES_PATH).exists():
    print(f'Loading cached traces from {TRACES_PATH}')
    traces = load_traces(TRACES_PATH)
else:
    print('Generating traces with real CompilerGym...')
    t0 = time.time()
    traces = generate_compiler_traces(
        benchmarks=CONFIG['benchmarks'],
        episodes_per_benchmark=CONFIG['episodes_per_benchmark'],
        max_steps_per_episode=CONFIG['max_steps_per_episode'],
        use_heuristic=False,
        output_path=TRACES_PATH,
        seed=CONFIG['seed'],
    )
    print(f'Generated {len(traces)} traces in {time.time()-t0:.0f}s')

print(f'Total trace records: {len(traces)}')

## Train TRM Model

In [ ]:
from torch.utils.data import DataLoader
from trm_compiler.model import TinyPassOrderingRefiner
from trm_compiler.training import train_one_epoch

dataset = CompilerTraceDataset(traces)
dataloader = DataLoader(
    dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=2, drop_last=True,
    pin_memory=(DEVICE == 'cuda'),
)

model = TinyPassOrderingRefiner(
    observation_dim=56,
    latent_dim=CONFIG['latent_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    num_passes=NUM_PASSES,
    n_recursions=CONFIG['n_recursions'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters')
print(f'Dataset: {len(dataset):,} records, {len(dataloader)} batches/epoch')

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

history = {k: [] for k in ['epoch', 'total_loss', 'pass_loss', 'value_loss', 'halt_loss', 'feasibility_loss']}
best_loss = float('inf')

print(f'\n{"Epoch":>5s} {"Total":>8s} {"Pass":>8s} {"Value":>8s} {"Halt":>8s}')
print('-' * 42)

for epoch in range(CONFIG['epochs']):
    t0 = time.time()
    losses = train_one_epoch(model, dataloader, optimizer, DEVICE)
    scheduler.step()

    for k in history:
        if k == 'epoch':
            history[k].append(epoch + 1)
        else:
            history[k].append(losses[k])

    print(f'{epoch+1:5d} {losses["total_loss"]:8.4f} {losses["pass_loss"]:8.4f} '
          f'{losses["value_loss"]:8.4f} {losses["halt_loss"]:8.4f}  ({time.time()-t0:.1f}s)')

    if losses['total_loss'] < best_loss:
        best_loss = losses['total_loss']
        torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG},
                   str(OUTPUT_DIR / 'trm_model_best.pt'))

torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG},
           str(OUTPUT_DIR / 'trm_model_final.pt'))
print(f'Best loss: {best_loss:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['epoch'], history['total_loss'], 'b-', lw=2)
ax1.set(xlabel='Epoch', ylabel='Loss (log)', title='Total Loss', yscale='log')
ax1.grid(True, alpha=0.3)

for name in ['pass_loss', 'value_loss', 'halt_loss', 'feasibility_loss']:
    ax2.plot(history['epoch'], history[name], label=name.replace('_', ' ').title(), lw=2)
ax2.set(xlabel='Epoch', ylabel='Loss (log)', title='Component Losses', yscale='log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate — Synthetic Environment (Quick Check)

In [ ]:
from trm_compiler.baselines import run_full_benchmark

model.eval()
run_full_benchmark(
    model,
    benchmarks=CONFIG['eval_benchmarks'],
    max_steps=CONFIG['eval_max_steps'],
    device=DEVICE,
    seed=CONFIG['seed'],
)

## Evaluate — Real LLVM (CompilerGym)

Compares LLVM -Oz, -O3, Random(100), and TRM on real LLVM passes.

In [ ]:
from trm_compiler.eval_real_llvm import (
    evaluate_model_on_real_llvm, print_results_table,
)

real_results = evaluate_model_on_real_llvm(
    model,
    benchmarks=CONFIG['eval_benchmarks'],
    max_steps=CONFIG['eval_max_steps'],
    device=DEVICE,
    num_random_trials=100,
    seed=CONFIG['seed'],
)

print_results_table(real_results, CONFIG['eval_benchmarks'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

benchmarks = CONFIG['eval_benchmarks']
algorithms = sorted(set(a for br in real_results.values() for a in br.keys()))
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(benchmarks))
width = 0.8 / len(algorithms)

for ax_idx, (key, ylabel, title) in enumerate([
    ('total_reward', 'Total Reward', 'Reward by Algorithm (Real LLVM)'),
    ('reduction_pct', 'Reduction %', 'Instruction Reduction (Real LLVM)'),
]):
    for i, algo in enumerate(algorithms):
        vals = [real_results[b].get(algo, {}).get(key, 0) for b in benchmarks]
        if key == 'reduction_pct':
            vals = [v * 100 for v in vals]
        axes[ax_idx].bar(x + i * width, vals, width, label=algo,
                         color=colors[i % len(colors)], alpha=0.85)
    axes[ax_idx].set(xlabel='Benchmark', ylabel=ylabel, title=title)
axes[ax_idx].set_xticks(x + width * (len(algorithms) - 1) / 2)
axes[ax_idx].set_xticklabels(benchmarks, rotation=45, ha='right')
axes[ax_idx].legend()
axes[ax_idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Detailed Rollout — Watch TRM on Real LLVM

In [ ]:
from trm_compiler.env_wrapper import CompilerGymWrapper, pass_id_to_name
from trm_compiler.model import encode_schedule

DETAIL_BENCH = 'qsort'
wrapper = CompilerGymWrapper(benchmark_id=f'cbench-v1/{DETAIL_BENCH}')
wrapper.open()
obs, initial_inst = wrapper.reset()

model.eval()
y, z = model.init_latent(1, DEVICE)
feedback_tensor = torch.zeros(1, 4, device=DEVICE)
applied_passes = []
total_reward = 0.0

print(f'{"Step":>4s} {"Pass":<28s} {"Prev":>6s} → {"New":>6s}  {"Reward":>8s}  {"Total":>8s}')
print('-' * 75)

for step in range(CONFIG['eval_max_steps']):
    schedule_vec = encode_schedule(step, CONFIG['eval_max_steps'], applied_passes)
    obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    sch_t = torch.tensor(schedule_vec, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        out = model(obs_t, sch_t, feedback_tensor, y, z)
    y, z = out['y'], out['z']

    halt_prob = torch.sigmoid(out['halt_logit']).item()
    if halt_prob > 0.5 and step >= 5:
        print(f'  HALT (prob={halt_prob:.3f})')
        break

    logits = out['pass_logits'].squeeze(0) / 0.8
    for pp in applied_passes[-3:]:
        logits[pp] -= 2.0
    pass_id = torch.multinomial(torch.softmax(logits, dim=-1).clamp(min=1e-6), 1).item()

    prev_inst = wrapper._current_inst_count
    obs, feedback, done, info = wrapper.step(pass_id)
    total_reward += feedback.reward
    applied_passes.append(pass_id)
    feedback_tensor = torch.tensor(feedback.encode(), dtype=torch.float32, device=DEVICE).unsqueeze(0)

    print(f'{step:4d} {pass_id_to_name(pass_id):<28s} {prev_inst:6d} → {wrapper._current_inst_count:6d}  '
          f'{feedback.reward:+8.4f}  {total_reward:+8.4f}')

    if done:
        break

wrapper.close()
print(f'\n{initial_inst} → {wrapper._current_inst_count} inst  ({(1 - wrapper._current_inst_count/initial_inst)*100:.1f}% reduction)')

In [ ]:
import json

# Save results
out = {}
for b, algos in real_results.items():
    out[b] = {a: {k: v for k, v in r.items() if k != 'steps'} for a, r in algos.items()}
with open(str(OUTPUT_DIR / 'eval_results.json'), 'w') as f:
    json.dump(out, f, indent=2)

print('Saved to', OUTPUT_DIR / 'eval_results.json')
for p in sorted(OUTPUT_DIR.iterdir()):
    sz = p.stat().st_size
    print(f'  {p.name:35s} {sz/1e6:.1f} MB' if sz > 1e6 else f'  {p.name:35s} {sz/1e3:.0f} KB')